# Cloud Provider Analytics — MVP técnico (Segunda Entrega)

**Pipeline:** `landing → Bronze → Silver → Gold → Serving (Cassandra/AstraDB)` con PySpark + Structured Streaming en Colab.

**Grupo:** Francisco Cavanna · Pedro Welsh · Tommy Wasserman

Este notebook corre de punta a punta. Las salidas de cada celda son la evidencia que pide la consigna (conteos, muestras de quarantine, resultados de las queries CQL, check de idempotencia).



## 0. Setup

Instalamos PySpark (pineado a 3.5.1, que es con lo que probamos) y el driver de Cassandra. Después montamos Drive — lo usamos como data lake (Landing + Bronze/Silver/Gold en Parquet + checkpoints del stream, que es lo que nos deja retomar si Colab reinicia la sesión).

In [ ]:
!pip install -q pyspark==3.5.1 cassandra-driver

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 27.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import shutil, os, glob
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               IntegerType, BooleanType, DateType)
BASE    = "/content/datalake"
LANDING = f"{BASE}/landing"
BRONZE  = f"{BASE}/bronze"
SILVER  = f"{BASE}/silver"
GOLD    = f"{BASE}/gold"
QUAR    = f"{BASE}/quarantine"
CKPT    = f"{BASE}/_checkpoints"
os.makedirs(BASE, exist_ok=True)

### Cargar el dataset a Landing

Subimos `cloud_provider_challenge_dataset_v1.zip` a nuestros Drives y lo descomprimimos dentro del lake.

In [14]:
ZIP_PATH = "/content/drive/MyDrive/cloud_provider_challenge_dataset_v1.zip"
if not os.path.exists(f"{LANDING}/customers_orgs.csv"):
    import zipfile
    tmp = "/content/_ds_tmp"
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(tmp)
    src = f"{tmp}/datalake/landing"
    os.makedirs(LANDING, exist_ok=True)
    for item in os.listdir(src):
        s = f"{src}/{item}"; d = f"{LANDING}/{item}"
        if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
        else: shutil.copy2(s, d)
print("Landing listo:", sorted(os.listdir(LANDING)))

Landing listo: ['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'usage_events_stream', 'users.csv']


In [15]:
spark = (SparkSession.builder.appName("cloud_analytics_mvp").master("local[*]")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


## 1. Batch → Bronze (maestros)

Leemos los CSV con **esquema explícito** ), agregamos las columnas técnicas (`ingest_ts`, `source_file`) y deduplicamos por la clave natural de cada tabla.

Cargamos los 3 maestros que pide el MVP: `customers_orgs`, `users`, `billing_monthly`.

**Sobre el particionado:** solo particionamos `billing_monthly` por `month`. A `customers_orgs` y `users` **no** los particionamos a propósito — son tablas chicas (80 y 800 filas), particionarlas genera muchos archivitos y solo agrega overhead. Por lo aprendido en la materia, particionar por particionar es mala práctica.

In [16]:
orgs_schema = StructType([
    StructField("org_id", StringType()), StructField("org_name", StringType()),
    StructField("industry", StringType()), StructField("hq_region", StringType()),
    StructField("plan_tier", StringType()), StructField("is_enterprise", StringType()),
    StructField("signup_date", DateType()), StructField("sales_rep", StringType()),
    StructField("lifecycle_stage", StringType()), StructField("marketing_source", StringType()),
    StructField("nps_score", DoubleType()),
])
users_schema = StructType([
    StructField("user_id", StringType()), StructField("org_id", StringType()),
    StructField("email", StringType()), StructField("role", StringType()),
    StructField("active", BooleanType()), StructField("created_at", DateType()),
    StructField("last_login", DateType()),
])
billing_schema = StructType([
    StructField("invoice_id", StringType()), StructField("org_id", StringType()),
    StructField("month", DateType()), StructField("subtotal", DoubleType()),
    StructField("credits", DoubleType()), StructField("taxes", DoubleType()),
    StructField("currency", StringType()), StructField("exchange_rate_to_usd", DoubleType()),
])

def read_master(name, schema):
    return (spark.read.option("header", True).schema(schema)
            .csv(f"{LANDING}/{name}.csv")
            .withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_path")))

orgs    = read_master("customers_orgs", orgs_schema).dropDuplicates(["org_id"])
users   = read_master("users", users_schema).dropDuplicates(["user_id"])
billing = read_master("billing_monthly", billing_schema).dropDuplicates(["invoice_id"])

orgs.write.mode("overwrite").parquet(f"{BRONZE}/customers_orgs")
users.write.mode("overwrite").parquet(f"{BRONZE}/users")
billing.write.mode("overwrite").partitionBy("month").parquet(f"{BRONZE}/billing_monthly")

print(f"orgs={orgs.count()}  users={users.count()}  billing={billing.count()}")

orgs=80  users=800  billing=240


## 2. Streaming → Bronze (usage events)

Structured Streaming leyendo el directorio `usage_events_stream/` con **esquema explícito**, `withWatermark`, dedup por `event_id` y checkpoint habilitado. Usamos `trigger(availableNow=True)` con `maxFilesPerTrigger=20`: drena los 120 archivos en micro-lotes (simulando el stream) y frena.

**Detalle importante sobre el watermark:** los 120 archivos están fragmentados en **orden aleatorio** — un mismo micro-lote trae eventos de julio y de agosto mezclados. Con un watermark chico (ej. 2 horas), apenas el stream ve un evento del 31/08 el watermark salta a esa fecha y **descarta como "late data" todos los eventos de julio** que lleguen después. Con un watermark de 2h perdíamos ~2/3 de los eventos.

La causa de fondo: un watermark asume que el tiempo-de-evento sigue al de llegada (stream ordenado). Este feed sintético **no está ordenado**, así que el watermark tiene que ser más ancho que el desorden máximo, que acá es ~el span completo (~60 días). En un stream real ordenado, un watermark de 2h alcanzaría y los stragglers más tardíos se descartarían.

**`value` lo dejamos como string en Bronze** a propósito: a veces viene número y a veces string. La consigna pide "castea con fallback", así que el casteo cuidadoso lo hacemos en Silver, no acá.

In [17]:
events_schema = StructType([
    StructField("event_id", StringType()), StructField("timestamp", StringType()),
    StructField("org_id", StringType()), StructField("resource_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("metric", StringType()),
    StructField("value", StringType()),
    StructField("unit", StringType()), StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()), StructField("genai_tokens", DoubleType()),
])

stream = (spark.readStream.format("json").schema(events_schema)
          .option("maxFilesPerTrigger", 20)
          .load(f"{LANDING}/usage_events_stream/"))

stream = (stream
          .withColumn("event_time", F.to_timestamp("timestamp"))
          .withColumn("usage_date", F.to_date("event_time"))
          .withColumn("ingest_ts", F.current_timestamp())
          .withColumn("source_file", F.col("_metadata.file_path"))

          .withWatermark("event_time", "60 days")
          .dropDuplicates(["event_id"]))

q = (stream.writeStream.format("parquet")
     .option("checkpointLocation", f"{CKPT}/bronze_events")
     .option("path", f"{BRONZE}/usage_events")
     .partitionBy("usage_date")
     .trigger(availableNow=True)
     .start())
q.awaitTermination()
print("streaming terminado")

streaming terminado


In [21]:
raw_count = (spark.read.format("json").schema(events_schema)
             .load(f"{LANDING}/usage_events_stream/").count())
bronze_events = spark.read.parquet(f"{BRONZE}/usage_events")
print(f"eventos RAW en landing: {raw_count}")
print(f"eventos en Bronze (streaming): {bronze_events.count()}")
bronze_events.groupBy("schema_version").count().orderBy("schema_version").show()

eventos RAW en landing: 43200
eventos en Bronze (streaming): 43200
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+



## 3. Silver (limpieza + join + features + calidad + quarantine)



1. **Conformance:** `trim` + `lower` en los campos categóricos (`service`, `region`, `metric`).
2. **Casteo de `value` con fallback:** `value` → double; si era no-nulo pero no parsea, queda null y lo marcamos con `value_cast_failed` (en esta data los strings son numéricos limpios, así que da 0, pero la regla está).
3. **Schema v1/v2:** los eventos v1 (anteriores al 18/07) no traían `carbon_kg` ni `genai_tokens`. Como los declaramos en el esquema explícito, ya quedan como null retroactivo — sin romper nada.
4. **3 reglas de calidad activas** + quarantine:
   - **R1** — `event_id` no nulo y único. (El dedup ya lo hizo Bronze; acá marcamos nulos.)
   - **R2** — `cost_usd_increment ≥ -0.01`. Los negativos por debajo de -0.01 → **quarantine**. Los positivos > p99×3 → se quedan pero con flag `is_cost_spike`.
   - **R3** — `unit` no nulo cuando `value` existe. En vez de tirar el evento (¡el costo es válido y lo perderíamos del mart!), **imputamos** la unidad desde el `metric` y lo marcamos con `unit_imputed`. La consigna permite imputación.
5. **Enriquecimiento:** `broadcast join` con `customers_orgs` (es chica, 80 filas → se replica en memoria, sin shuffle) para sumar `plan_tier`, `industry`, `hq_region`.

In [22]:
ev = spark.read.parquet(f"{BRONZE}/usage_events")

ev = (ev.withColumn("service", F.lower(F.trim("service")))
        .withColumn("region", F.lower(F.trim("region")))
        .withColumn("metric", F.lower(F.trim("metric"))))

ev = (ev.withColumn("value_double", F.col("value").cast("double"))
        .withColumn("value_cast_failed",
                    F.col("value").isNotNull() & F.col("value_double").isNull()))

ev = ev.withColumn("r1_bad_id", F.col("event_id").isNull())

p99 = ev.approxQuantile("cost_usd_increment", [0.99], 0.001)[0]
spike_thr = p99 * 3
ev = (ev.withColumn("r2_bad_cost", F.col("cost_usd_increment") < F.lit(-0.01))
        .withColumn("is_cost_spike", F.col("cost_usd_increment") > F.lit(spike_thr)))

unit_map = F.create_map(
    F.lit("requests"), F.lit("count"),
    F.lit("cpu_hours"), F.lit("hours"),
    F.lit("storage_gb_hours"), F.lit("gb_hours"))
ev = (ev.withColumn("unit_imputed",
                    F.col("unit").isNull() & F.col("value_double").isNotNull())
        .withColumn("unit", F.when(F.col("unit_imputed"), unit_map[F.col("metric")])
                             .otherwise(F.col("unit"))))

ev = ev.cache()
ev.count()

quarantine = (ev.filter(F.col("r1_bad_id") | F.col("r2_bad_cost"))
                .withColumn("quarantine_reason",
                            F.when(F.col("r1_bad_id"), F.lit("event_id_null"))
                             .when(F.col("r2_bad_cost"), F.lit("cost_below_-0.01"))))
clean = ev.filter(~(F.col("r1_bad_id") | F.col("r2_bad_cost")))

org_dim = spark.read.parquet(f"{BRONZE}/customers_orgs").select(
    "org_id", "plan_tier", "industry", "hq_region")
silver = clean.join(F.broadcast(org_dim), "org_id", "left")

silver.coalesce(1).write.mode("overwrite").partitionBy("usage_date").parquet(f"{SILVER}/usage_events")
quarantine.coalesce(1).write.mode("overwrite").parquet(f"{QUAR}/usage_events")

print(f"clean={clean.count()}  quarantine={quarantine.count()}  (p99={p99:.3f}, spike_thr={spike_thr:.2f})")
print(f"unit_imputed={ev.filter('unit_imputed').count()}  spikes={ev.filter('is_cost_spike').count()}  value_cast_failed={ev.filter('value_cast_failed').count()}")

clean=42989  quarantine=211  (p99=16.675, spike_thr=50.03)
unit_imputed=2038  spikes=54  value_cast_failed=0


In [23]:
print("Muestras de quarantine:")
(quarantine.select("event_id","org_id","service","cost_usd_increment","quarantine_reason")
 .show(8, truncate=False))

Muestras de quarantine:
+----------------+------------+----------+------------------+-----------------+
|event_id        |org_id      |service   |cost_usd_increment|quarantine_reason|
+----------------+------------+----------+------------------+-----------------+
|evt_kp4vwxvp8qsw|org_chj755nf|compute   |-0.2877           |cost_below_-0.01 |
|evt_fac9pru3xlpd|org_dhylurtp|database  |-0.5116           |cost_below_-0.01 |
|evt_kwcwbndmwq2t|org_w3zp08j3|database  |-0.7245           |cost_below_-0.01 |
|evt_j4eh0lzqk5f8|org_253j2d54|analytics |-7.4814           |cost_below_-0.01 |
|evt_wjaqteycon65|org_teiyzcot|networking|-1.3905           |cost_below_-0.01 |
|evt_3rygp8arcgcj|org_pbhsahxt|genai     |-2.7801           |cost_below_-0.01 |
|evt_ttdp73k5rysi|org_1n13jcat|storage   |-0.0963           |cost_below_-0.01 |
|evt_vx4t19bkerlv|org_sg65kxvf|database  |-6.8091           |cost_below_-0.01 |
+----------------+------------+----------+------------------+-----------------+
only showing top

## 4. Gold — mart FinOps `org_daily_usage_by_service`

Agregamos a grano **org × servicio × día**. Métricas: costo diario, requests, cpu_hours, storage_gb_hours, tokens GenAI, carbon, conteo de eventos, flag de anomalía, y arrastramos `plan_tier`/`industry` del join (así la tabla sirve para cortar por plan en FinOps).

Esta tabla está diseñada **query-first** para responder las dos consultas que pide el MVP (costos/requests por org-servicio-rango, y top-N servicios por costo en 14 días).

In [24]:
silver = spark.read.parquet(f"{SILVER}/usage_events")
gold = (silver.groupBy("org_id", "usage_date", "service")
        .agg(
            F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
            F.sum(F.when(F.col("metric")=="requests", F.col("value_double"))).alias("requests"),
            F.sum(F.when(F.col("metric")=="cpu_hours", F.col("value_double"))).alias("cpu_hours"),
            F.sum(F.when(F.col("metric")=="storage_gb_hours", F.col("value_double"))).alias("storage_gb_hours"),
            F.round(F.sum("genai_tokens"), 2).alias("genai_tokens"),
            F.round(F.sum("carbon_kg"), 4).alias("carbon_kg"),
            F.count("*").alias("event_count"),
            F.max(F.col("is_cost_spike").cast("int")).alias("has_anomaly"),
            F.first("plan_tier", ignorenulls=True).alias("plan_tier"),
            F.first("industry", ignorenulls=True).alias("industry"),
        ))
gold.write.mode("overwrite").partitionBy("usage_date").parquet(f"{GOLD}/org_daily_usage_by_service")
print(f"filas Gold (org x service x day): {gold.count()}")
gold.orderBy("org_id","usage_date","service").show(8, truncate=False)

tot_silver = silver.agg(F.round(F.sum("cost_usd_increment"),2)).collect()[0][0]
tot_gold   = gold.agg(F.round(F.sum("daily_cost_usd"),2)).collect()[0][0]
print(f"CHECK cost total -> silver={tot_silver}  gold={tot_gold}  (deben coincidir)")

filas Gold (org x service x day): 11050
+------------+----------+---------+--------------+--------+---------+----------------+------------+---------+-----------+-----------+---------+--------+
|org_id      |usage_date|service  |daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|event_count|has_anomaly|plan_tier|industry|
+------------+----------+---------+--------------+--------+---------+----------------+------------+---------+-----------+-----------+---------+--------+
|org_0lvsnujz|2025-07-03|analytics|8.8739        |116.0   |4.6168   |12.7808         |NULL        |NULL     |3          |0          |standard |Retail  |
|org_0lvsnujz|2025-07-03|database |0.5723        |NULL    |1.7515   |14.7891         |NULL        |NULL     |3          |0          |standard |Retail  |
|org_0lvsnujz|2025-07-03|storage  |5.5542        |255.0   |NULL     |4.6668          |NULL        |NULL     |3          |0          |standard |Retail  |
|org_0lvsnujz|2025-07-04|analytics|3.2915 

## 5. Serving en AstraDB (Cassandra)


In [25]:
import requests, os

ASTRA_TOKEN = "AstraCS:vTmhitOuewnGHUJPfGbnuCOs:095b1212c91df22c082f4f19d15a739f251f685c53c7094a7e0513b426a16f53"
DB_ID       = "de83959c-7447-4099-bb0f-decf663f92b0"   # <-- el ID que se ve en la consola de Astra
BUNDLE_PATH = "/content/secure-connect-cloud-analytics.zip"


resp = requests.post(
    f"https://api.astra.datastax.com/v2/databases/{DB_ID}/secureBundleURL",
    headers={"Authorization": f"Bearer {ASTRA_TOKEN}", "Content-Type": "application/json"},
    json={},
)
resp.raise_for_status()
data = resp.json()
bundle_url = data[0]["downloadURL"] if isinstance(data, list) else data["downloadURL"]

r = requests.get(bundle_url)
r.raise_for_status()
with open(BUNDLE_PATH, "wb") as f:
    f.write(r.content)

print(f"bundle descargado: {os.path.getsize(BUNDLE_PATH)} bytes -> {BUNDLE_PATH}")

bundle descargado: 12312 bytes -> /content/secure-connect-cloud-analytics.zip


In [26]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.concurrent import execute_concurrent_with_args

KEYSPACE = "cloud_analytics"

cloud_config = {"secure_connect_bundle": BUNDLE_PATH}
auth = PlainTextAuthProvider("token", ASTRA_TOKEN)
cluster = Cluster(cloud=cloud_config, auth_provider=auth)
session = cluster.connect()
session.set_keyspace(KEYSPACE)
print("conectado a AstraDB, keyspace =", KEYSPACE)

conectado a AstraDB, keyspace = cloud_analytics


### Tabla modelada query-first

`PRIMARY KEY ((org_id), usage_date, service)`:
- **partition key = `org_id`** → cada org es una partición. Las dos queries filtran por una sola org → pegan a una sola partición (sin `ALLOW FILTERING`, sin scatter-gather).
- **clustering = `usage_date, service`** → `usage_date` primero permite hacer rangos de fechas eficientes; `service` desempata dentro del día.

In [27]:
session.execute('''
CREATE TABLE IF NOT EXISTS org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    daily_cost_usd double,
    requests double,
    cpu_hours double,
    storage_gb_hours double,
    genai_tokens double,
    carbon_kg double,
    event_count int,
    has_anomaly int,
    plan_tier text,
    industry text,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date ASC, service ASC)
''')
print("tabla creada")

tabla creada


In [28]:
insert = session.prepare('''
INSERT INTO org_daily_usage_by_service
(org_id, usage_date, service, daily_cost_usd, requests, cpu_hours, storage_gb_hours,
 genai_tokens, carbon_kg, event_count, has_anomaly, plan_tier, industry)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
''')

gold = spark.read.parquet(f"{GOLD}/org_daily_usage_by_service")
rows = gold.collect()
params = [(r["org_id"], r["usage_date"], r["service"], r["daily_cost_usd"], r["requests"],
           r["cpu_hours"], r["storage_gb_hours"], r["genai_tokens"], r["carbon_kg"],
           r["event_count"], r["has_anomaly"], r["plan_tier"], r["industry"]) for r in rows]

results = execute_concurrent_with_args(session, insert, params, concurrency=50)
ok = sum(1 for s,_ in results if s)
print(f"filas upserteadas en Cassandra: {ok}/{len(params)}")

filas upserteadas en Cassandra: 11050/11050


### 5.1 — Query #1: costos y requests diarios por org y servicio en un rango de fechas

Consulta nativa de una sola partición con rango sobre `usage_date`. Elegimos una org con datos automáticamente.

In [29]:
import datetime as dt

sample_org = (gold.groupBy("org_id").count().orderBy(F.desc("count")).first()["org_id"])
max_date = gold.agg(F.max("usage_date")).collect()[0][0]
d_from = max_date - dt.timedelta(days=7)
print(f"org demo: {sample_org} | rango: {d_from} -> {max_date}\n")

q1 = session.prepare('''
SELECT usage_date, service, daily_cost_usd, requests
FROM org_daily_usage_by_service
WHERE org_id = ? AND usage_date >= ? AND usage_date <= ?
''')
for row in session.execute(q1, (sample_org, d_from, max_date)):
    print(f"{row.usage_date}  {row.service:12s}  cost={row.daily_cost_usd:8.3f}  requests={row.requests}")

org demo: org_53lc58dr | rango: 2025-08-24 -> 2025-08-31

2025-08-24  analytics     cost=  17.717  requests=259.0
2025-08-24  compute       cost=   8.503  requests=119.0
2025-08-24  database      cost=  12.941  requests=231.0
2025-08-25  analytics     cost=  22.351  requests=340.0
2025-08-25  database      cost=   6.230  requests=107.0
2025-08-25  networking    cost=   2.574  requests=248.0
2025-08-26  analytics     cost=  30.205  requests=335.0
2025-08-26  compute       cost=  18.511  requests=225.0
2025-08-26  database      cost=  24.976  requests=355.0
2025-08-26  genai         cost=   1.817  requests=None
2025-08-26  networking    cost=   1.192  requests=122.0
2025-08-26  storage       cost=   2.748  requests=121.0
2025-08-27  analytics     cost=  18.093  requests=226.0
2025-08-27  compute       cost=  42.189  requests=516.0
2025-08-27  database      cost=  10.634  requests=238.0
2025-08-27  genai         cost=  43.688  requests=349.0
2025-08-27  storage       cost=   0.277  reques

### 5.2 — Query #2: Top-N servicios por costo acumulado en los últimos 14 días (una org)

Traemos el slice de 14 días de esa org (una sola partición) y agregamos por servicio **del lado del cliente**, ordenamos y tomamos top-N.



In [30]:
from collections import defaultdict
d14 = max_date - dt.timedelta(days=14)
N = 5
agg = defaultdict(float)
for row in session.execute(q1, (sample_org, d14, max_date)):
    if row.daily_cost_usd: agg[row.service] += row.daily_cost_usd

top = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:N]
print(f"Top-{N} servicios por costo (últimos 14 días) — org {sample_org}:")
for i,(svc,cost) in enumerate(top,1):
    print(f"  {i}. {svc:12s}  ${cost:,.2f}")

Top-5 servicios por costo (últimos 14 días) — org org_53lc58dr:
  1. genai         $302.15
  2. analytics     $282.07
  3. compute       $247.93
  4. database      $182.78
  5. storage       $43.23


## 6. Idempotencia — reprocesar no duplica

Contamos las filas en Cassandra, volvemos a correr el upsert de Gold, y contamos de nuevo. Como los `INSERT` de Cassandra son **upserts por primary key**, re-ejecutar pisa la misma fila en vez de duplicar → el conteo no cambia. (Sumado al checkpoint del stream, que evita reprocesar archivos ya leídos.)



In [31]:
def count_table():
    return session.execute("SELECT COUNT(*) FROM org_daily_usage_by_service").one()[0]

before = count_table()
print("filas antes del re-run:", before)


execute_concurrent_with_args(session, insert, params, concurrency=50)

after = count_table()
print("filas después del re-run:", after)
print("IDEMPOTENTE ✓" if before == after else "PROBLEMA: se duplicó")

filas antes del re-run: 11050


filas después del re-run: 11050
IDEMPOTENTE ✓
